# Walkthrough Video to 3D Gaussian Splat, Nerfstudio + Splatfacto

Turn a **smartphone walkthrough video** of an indoor space into a downloadable
**3D Gaussian Splat** (`.ply` + `.splat`), fully inside Google Colab.

This notebook uses the **Nerfstudio + Splatfacto** pipeline (with the CUDA
`gsplat` backend). It was selected after comparing the strongest open-source
options available today:

| Pipeline | Why / why not |
|---|---|
| **Nerfstudio + Splatfacto (chosen)** | Actively maintained, best documented, fully automated `video -> poses -> splat -> .ply`, no pretrained weights, embeds `gsplat` MCMC + camera pose refinement. |
| gsplat standalone | Same engine, top quality, but you hand-manage COLMAP and data conventions. Nerfstudio gives the same engine with guardrails. |
| DN-Splatter | Best *indoor geometry/mesh* (depth+normal priors), but heavier install and more version pinning. Documented here as an optional upgrade. |
| OpenSplat | Easiest to build, but trails on densification/quality. Fallback only. |
| Original 3DGS / 2DGS | Superseded by gsplat on speed/memory/quality. |
| Postshot | Proprietary Windows GUI, not Colab. Excluded. |

**How the pipeline works**

1. **Frames** are extracted from your video.
2. **Structure-from-Motion** (COLMAP, optionally hloc/SuperPoint+SuperGlue)
   recovers camera poses and a sparse point cloud, the modern replacement for
   manual camera calibration.
3. **Splatfacto** optimizes a set of 3D Gaussians against your images
   (per-scene, no learned weights required).
4. The result is **exported** as a lossless `.ply` and a web-ready `.splat`,
   then zipped for download.

> **What you do:** set a couple of options, run the cells top to bottom, upload
> your video when asked. Everything else is automatic.

---

### Before you start, enable the GPU

`Runtime -> Change runtime type -> Hardware accelerator -> GPU` (T4 works; L4 / A100
are faster and allow higher quality). Then run the cells in order.


## 1. Check the GPU and runtime

We confirm a CUDA GPU is present before doing anything expensive. If this cell
reports **no GPU**, fix the runtime type (see above) and re-run it.


In [ ]:
#@title Verify GPU { display-mode: "form" }
import subprocess, sys, textwrap

def banner(msg, ch="="):
    print(ch * 72); print(msg); print(ch * 72)

banner("GPU / RUNTIME CHECK")

try:
    out = subprocess.check_output(
        ["nvidia-smi",
         "--query-gpu=name,memory.total,driver_version",
         "--format=csv,noheader"],
        text=True).strip()
    name, mem, driver = [x.strip() for x in out.split(",")]
    print(f"  GPU            : {name}")
    print(f"  VRAM           : {mem}")
    print(f"  Driver         : {driver}")
    mem_gb = float(mem.split()[0]) / 1024.0
    GPU_MEM_GB = mem_gb
    if mem_gb < 14:
        print("\n  NOTE: <16 GB VRAM detected (typical of a free T4).")
        print("        Use the 'balanced' quality preset and keep frame count")
        print("        moderate to avoid out-of-memory errors.")
    else:
        print("\n  Plenty of VRAM, you can use the 'high' or 'max' quality preset.")
    print("\n  GPU OK.")
except Exception as e:
    GPU_MEM_GB = 0.0
    banner("NO GPU DETECTED", "!")
    print(textwrap.dedent("""
        This notebook requires a CUDA GPU.

        Fix: Runtime -> Change runtime type -> Hardware accelerator -> GPU,
        then re-run this cell. If you just changed it, the runtime restarts
        and you must run this cell again.
    """))
    raise SystemExit("Stop here until a GPU is available.")

print(f"\n  Python: {sys.version.split()[0]}")

## 2. Choose your settings

These options control quality vs. speed. The defaults are good for a first run.
You can change them and re-run individual stages later.

- **quality_preset**
  - `balanced` , Splatfacto default. Fastest, ~6 GB VRAM. Good for free T4.
  - `high` (recommended) , `splatfacto-big` with quality tweaks + camera pose
    refinement. ~12 GB VRAM.
  - `max` , adds the `gsplat` **MCMC** densification strategy for the best
    detail (needs a recent nerfstudio; falls back automatically if unavailable).
- **train_iterations** , more iterations = higher quality, longer training.
- **target_frames** , how many frames to sample from the video. 100 to 300 is a
  good range for a walkthrough. More frames = better coverage but slower SfM.
- **sfm_tool**
  - `colmap` , classic SIFT SfM, reliable default.
  - `hloc` , SuperPoint + SuperGlue, more robust for shaky / low-texture phone
    video (slower, downloads small models).


In [ ]:
#@title Pipeline settings { display-mode: "form" }

quality_preset   = "high"   #@param ["balanced", "high", "max"]
train_iterations = 30000    #@param {type:"slider", min:5000, max:60000, step:1000}
target_frames    = 200      #@param {type:"slider", min:50, max:500, step:10}
sfm_tool         = "colmap" #@param ["colmap", "hloc"]
scene_name       = "my_scene" #@param {type:"string"}

import re, os
scene_name = re.sub(r"[^A-Za-z0-9_\-]", "_", scene_name) or "my_scene"

# Working directories used throughout the notebook.
WORK       = "/content/gsplat_project"
VIDEO_DIR  = f"{WORK}/video"
DATA_DIR   = f"{WORK}/processed/{scene_name}"   # nerfstudio dataset (transforms.json)
OUTPUT_DIR = f"{WORK}/outputs"                  # nerfstudio training outputs
EXPORT_DIR = f"{WORK}/exports/{scene_name}"     # final .ply / .splat / .zip
for d in (VIDEO_DIR, DATA_DIR, OUTPUT_DIR, EXPORT_DIR):
    os.makedirs(d, exist_ok=True)

print("Settings")
print("  quality_preset  :", quality_preset)
print("  train_iterations:", train_iterations)
print("  target_frames   :", target_frames)
print("  sfm_tool        :", sfm_tool)
print("  scene_name      :", scene_name)
print("\nPaths")
print("  data (poses)    :", DATA_DIR)
print("  training output :", OUTPUT_DIR)
print("  exports         :", EXPORT_DIR)

## 3. Install system tools (COLMAP, FFmpeg)

Installs the operating-system packages the pipeline needs. This runs quietly and
prints a short status line for each package. Takes ~1 to 2 minutes.


In [ ]:
#@title Install COLMAP + FFmpeg + helpers { display-mode: "form" }
import subprocess, sys, os, time

def run(cmd, desc=None, log_path=None, ok_msg=None):
    """Run a shell command, show a compact live status line, raise on failure."""
    if desc:
        print(f"-> {desc}")
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail, logf = [], open(log_path, "w") if log_path else None
    for line in proc.stdout:
        if logf: logf.write(line)
        tail.append(line.rstrip())
        if len(tail) > 400: tail = tail[-400:]
        s = line.rstrip()[:100]
        sys.stdout.write("\r   " + s.ljust(102)); sys.stdout.flush()
    proc.wait()
    if logf: logf.close()
    sys.stdout.write("\r" + " " * 106 + "\r"); sys.stdout.flush()
    if proc.returncode != 0:
        print(f"   FAILED: {desc or cmd}")
        print("   --- last lines " + "-" * 40)
        for l in tail[-25:]:
            print("   " + l)
        raise RuntimeError(f"Command failed (exit {proc.returncode}): {cmd}")
    if ok_msg: print(f"   {ok_msg}")
    elif desc: print(f"   done")
    return "\n".join(tail)

print("Installing system packages (quiet)...\n")
run("apt-get -qq update", "apt update")
run("DEBIAN_FRONTEND=noninteractive apt-get -qq install -y "
    "colmap ffmpeg xvfb", "colmap + ffmpeg + xvfb")

# Verify
for tool, args in [("colmap", "-h"), ("ffmpeg", "-version")]:
    try:
        v = subprocess.run([tool, *args.split()], capture_output=True, text=True)
        first = (v.stdout or v.stderr).splitlines()[0][:60]
        print(f"   {tool:8s}: {first}")
    except Exception as e:
        raise RuntimeError(f"{tool} not found after install: {e}")

print("\nSystem tools ready.")

## 4. Install Nerfstudio + gsplat

Installs the Python reconstruction stack. `gsplat` compiles its CUDA kernels the
first time it is used, so this cell also does a quick warm-up import to surface
any build problems early rather than mid-training.

This is the longest install step (~3 to 6 minutes). Colab base images change over
time; if an install ever fails here, the error tail is printed so you can see
exactly which package is unhappy.


In [ ]:
#@title Install Nerfstudio + gsplat { display-mode: "form" }
import os, subprocess, sys

# Help gsplat build only for this GPU's architecture (faster compile).
try:
    import torch
    cap = torch.cuda.get_device_capability()
    os.environ["TORCH_CUDA_ARCH_LIST"] = f"{cap[0]}.{cap[1]}"
    print("Detected torch", torch.__version__, "CUDA", torch.version.cuda,
          "arch", os.environ["TORCH_CUDA_ARCH_LIST"])
except Exception as e:
    print("Could not import torch yet:", e)

print("\nInstalling nerfstudio (this pulls gsplat). Please wait...\n")
# nerfstudio pulls a compatible gsplat and viser. We avoid tiny-cuda-nn because
# Splatfacto does not need it (only NeRF methods do), which removes the most
# fragile part of the install.
run("pip -q install nerfstudio", "nerfstudio + gsplat + dependencies",
    log_path="/content/install_nerfstudio.log")

# Extra helpers for SfM robustness (hloc) and export conversion.
run("pip -q install 'pycolmap>=0.6.1' || true", "pycolmap (optional)")

print("\nWarming up gsplat CUDA kernels (first compile can take 1 to 3 min)...")
warm = subprocess.run(
    [sys.executable, "-c",
     "import torch, gsplat, warnings; warnings.filterwarnings('ignore');"
     "print('gsplat', gsplat.__version__);"
     "from gsplat import rasterization; print('gsplat import OK')"],
    capture_output=True, text=True)
print(warm.stdout.strip())
if warm.returncode != 0:
    print("   gsplat warm-up reported:\n", warm.stderr[-1500:])
    print("\n   If this is a compile error, re-run this cell once (kernels cache),")
    print("   or restart the runtime and run cells 1-4 again.")
else:
    print("   gsplat ready.")

# Verify the nerfstudio CLI is on PATH.
cli = subprocess.run(["ns-train", "--help"], capture_output=True, text=True)
if cli.returncode == 0:
    print("\nNerfstudio CLI OK  (ns-process-data / ns-train / ns-export available)")
else:
    print("\nns-train not found on PATH:\n", cli.stderr[-800:])
    raise RuntimeError("Nerfstudio CLI not available, check the install log above.")

## 5. Upload your walkthrough video

Run the cell and pick your video (`.mp4`, `.mov`, `.m4v`, `.avi`, `.mkv`, `.webm`).

**Tips for a good capture**

- Walk slowly and smoothly; avoid motion blur.
- Keep good, even lighting; avoid moving people/objects in frame.
- Overlap your views; cover walls, corners, and the floor.
- 30 to 90 seconds is usually plenty.

If your video is large, uploading through the browser can be slow. As an
alternative you can mount Google Drive (see the commented lines) and point the
notebook at a file there.


In [ ]:
#@title Upload video (or use a Drive path) { display-mode: "form" }
import os, glob, shutil, subprocess, json

# --- Option A: mount Google Drive and set a path instead of uploading ---------
# from google.colab import drive
# drive.mount('/content/drive')
# drive_video = '/content/drive/MyDrive/my_walkthrough.mp4'   # <- set this
drive_video = ""  #@param {type:"string"}

VIDEO_EXTS = (".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm")
VIDEO_PATH = None

if drive_video.strip():
    src = drive_video.strip()
    if not os.path.isfile(src):
        raise FileNotFoundError(f"drive_video not found: {src}")
    VIDEO_PATH = os.path.join(VIDEO_DIR, os.path.basename(src))
    shutil.copy(src, VIDEO_PATH)
    print("Using Drive video:", VIDEO_PATH)
else:
    from google.colab import files
    print("Choose your walkthrough video...")
    up = files.upload()
    if not up:
        raise RuntimeError("No file uploaded.")
    fn = list(up.keys())[0]
    if not fn.lower().endswith(VIDEO_EXTS):
        raise ValueError(f"'{fn}' is not a recognized video type {VIDEO_EXTS}.")
    VIDEO_PATH = os.path.join(VIDEO_DIR, fn)
    with open(VIDEO_PATH, "wb") as f:
        f.write(up[fn])

# Probe the video so the user gets immediate, readable feedback.
def ffprobe(path):
    q = ("ffprobe -v error -select_streams v:0 -show_entries "
         "stream=width,height,r_frame_rate,nb_frames:format=duration "
         f"-of json '{path}'")
    r = subprocess.run(q, shell=True, capture_output=True, text=True)
    return json.loads(r.stdout) if r.returncode == 0 else {}

info = ffprobe(VIDEO_PATH)
size_mb = os.path.getsize(VIDEO_PATH) / 1e6
print("\nVideo ready")
print(f"  file    : {os.path.basename(VIDEO_PATH)}  ({size_mb:.1f} MB)")
try:
    st = info["streams"][0]
    dur = float(info.get("format", {}).get("duration", 0))
    num, den = (st.get("r_frame_rate", "0/1").split("/") + ["1"])[:2]
    fps = float(num) / float(den) if float(den) else 0
    print(f"  size    : {st.get('width')}x{st.get('height')}")
    print(f"  duration: {dur:.1f} s   (~{fps:.1f} fps source)")
    if dur and dur < 5:
        print("  WARNING: very short video (<5s), reconstruction may be poor.")
except Exception:
    print("  (could not probe details, but the file will still be processed)")

## 6. Recover camera poses (Structure-from-Motion)

This is the modern replacement for manual camera calibration. `ns-process-data`
extracts frames from your video and runs SfM to recover, for every frame, where
the camera was and where it pointed, plus a sparse 3D point cloud used to
initialize the Gaussians.

This is usually the **slowest** stage (several minutes to ~20+ min depending on
frame count, resolution, and GPU). Progress is summarized; the full log is saved
to a file. At the end we verify that a valid `transforms.json` with registered
frames was produced, if SfM fails to register your frames, you will get a clear
message rather than a broken training run.


In [ ]:
#@title Run SfM to get camera poses { display-mode: "form" }
import os, json, glob, shutil, subprocess

# Fresh dataset dir so re-runs do not mix old results.
if os.path.isdir(DATA_DIR) and os.listdir(DATA_DIR):
    shutil.rmtree(DATA_DIR); os.makedirs(DATA_DIR, exist_ok=True)

# COLMAP's GPU feature extraction needs an X server on headless Colab; xvfb-run
# provides a virtual one. hloc uses its own (GPU torch) matcher, no xvfb needed.
prefix = "" if sfm_tool == "hloc" else "xvfb-run -a "

cmd = (
    f"{prefix}ns-process-data video "
    f"--data '{VIDEO_PATH}' "
    f"--output-dir '{DATA_DIR}' "
    f"--num-frames-target {target_frames} "
    f"--sfm-tool {sfm_tool} "
    f"--matching-method {'sequential' if sfm_tool=='colmap' else 'exhaustive'} "
    "--verbose"
)
print("SfM command:\n ", cmd, "\n")
print("Working... (a compact status line updates below; full log is saved)\n")

log_path = "/content/ns_process.log"
try:
    run(cmd, "Structure-from-Motion (frames + poses + sparse points)",
        log_path=log_path, ok_msg="SfM finished")
except RuntimeError as e:
    print("\nSfM did not complete. Common causes and fixes:")
    print("  - Too little overlap / motion blur -> recapture, move slower.")
    print("  - Textureless walls -> try sfm_tool='hloc' in cell 2, re-run.")
    print("  - Too few frames registered -> raise target_frames, or shorten clip.")
    raise

# --- Verify the produced dataset -------------------------------------------
tj = os.path.join(DATA_DIR, "transforms.json")
if not os.path.isfile(tj):
    raise RuntimeError(f"No transforms.json at {tj}, SfM produced no valid poses.")

with open(tj) as f:
    T = json.load(f)
n_frames = len(T.get("frames", []))
n_imgs = len(glob.glob(os.path.join(DATA_DIR, "images", "*")))
has_pts = os.path.isfile(os.path.join(DATA_DIR, "sparse_pc.ply")) or \
          bool(glob.glob(os.path.join(DATA_DIR, "colmap", "sparse", "**", "points3D*"),
                         recursive=True))

print("\nDataset summary")
print(f"  registered frames : {n_frames}")
print(f"  extracted images  : {n_imgs}")
print(f"  sparse points     : {'yes' if has_pts else 'not found (splats will use random init)'}")

if n_frames < 20:
    print("\n  WARNING: very few frames were registered (<20). Quality will be low.")
    print("           Consider recapturing with more overlap, or sfm_tool='hloc'.")
else:
    print("\n  Poses look good. Ready to train.")

## 7. Train the Gaussian Splat

Splatfacto optimizes the 3D Gaussians against your images. There are no
pretrained weights, the model is fit to *your* scene from scratch, initialized
from the SfM points.

The command is built from your **quality_preset**:

- `balanced` , `splatfacto` defaults.
- `high` , `splatfacto-big` with a lower alpha-cull threshold, culling disabled
  after densification, and **camera pose refinement** (helps handheld phone
  video).
- `max` , adds the `gsplat` **MCMC** strategy for maximum detail.

Training prints periodic progress. Expect anywhere from ~10 minutes (balanced,
few frames) to ~45+ minutes (max, many frames). You can lower `train_iterations`
in cell 2 for a faster preview.


In [ ]:
#@title Train Splatfacto { display-mode: "form" }
import os, glob, subprocess, sys

# Base flags shared by all presets.
common = [
    f"--data '{DATA_DIR}'",
    f"--output-dir '{OUTPUT_DIR}'",
    f"--max-num-iterations {train_iterations}",
    "--viewer.quit-on-train-completion True",
    "--vis tensorboard",           # headless-friendly (no tunnel needed)
    f"--experiment-name {scene_name}",
]

if quality_preset == "balanced":
    method = "splatfacto"
    extra  = ["--pipeline.model.camera-optimizer.mode SO3xR3"]
elif quality_preset == "high":
    method = "splatfacto-big"
    extra  = [
        "--pipeline.model.cull_alpha_thresh 0.005",
        "--pipeline.model.continue_cull_post_densification False",
        "--pipeline.model.camera-optimizer.mode SO3xR3",
    ]
else:  # max
    method = "splatfacto"
    extra  = [
        "--pipeline.model.strategy mcmc",
        "--pipeline.model.cull_alpha_thresh 0.005",
        "--pipeline.model.camera-optimizer.mode SO3xR3",
    ]

def build_cmd(method, extra):
    return (f"ns-train {method} " + " ".join(common + extra) +
            " nerfstudio-data --data '%s'" % DATA_DIR)

cmd = build_cmd(method, extra)
print("Training command:\n ", cmd, "\n")
print("Training... (nerfstudio prints a periodic progress table)\n")

log_path = "/content/ns_train.log"
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
with open(log_path, "w") as logf:
    for line in proc.stdout:
        logf.write(line)
        # Surface only the informative lines to keep output readable.
        s = line.rstrip()
        if any(k in s for k in ("Step ", "ETA", "Loss", "PSNR", "iter",
                                "Error", "Traceback", "CUDA out of memory",
                                "Saving", "Printing profiling")):
            print(s[:120])
proc.wait()

if proc.returncode != 0:
    print("\nTraining failed.")
    if "max" == quality_preset:
        print("The 'max' (MCMC) preset needs a recent nerfstudio. Retrying with")
        print("the 'high' preset automatically...\n")
        cmd = build_cmd("splatfacto-big", [
            "--pipeline.model.cull_alpha_thresh 0.005",
            "--pipeline.model.continue_cull_post_densification False",
            "--pipeline.model.camera-optimizer.mode SO3xR3"])
        r = subprocess.run(cmd, shell=True)
        if r.returncode != 0:
            raise RuntimeError("Training failed, see /content/ns_train.log")
    else:
        # Show a hint for the most common failure.
        with open(log_path) as f:
            tail = f.read()[-1500:]
        if "out of memory" in tail.lower():
            print("CUDA out of memory. Fixes: use 'balanced' preset, lower")
            print("target_frames, or pick a bigger GPU (L4/A100).")
        print("\n--- log tail ---\n", tail)
        raise RuntimeError("Training failed, see /content/ns_train.log")

# Locate the trained config.yml that export will consume.
cfgs = sorted(glob.glob(os.path.join(OUTPUT_DIR, "**", "config.yml"),
                        recursive=True), key=os.path.getmtime)
if not cfgs:
    raise RuntimeError("Training finished but no config.yml was found.")
CONFIG_YML = cfgs[-1]
print("\nTraining complete.")
print("  config:", CONFIG_YML)

## 8. Export the Gaussian Splat

We export the trained model to a **`.ply`** (the lossless, highest-quality
Gaussian format, ingestible by SuperSplat, PlayCanvas, Three.js, Polycam, etc.)
and also generate a compact **`.splat`** file for lightweight web viewers.


In [ ]:
#@title Export .ply (+ .splat) { display-mode: "form" }
import os, glob, struct, numpy as np, subprocess

os.makedirs(EXPORT_DIR, exist_ok=True)

# --- 1. Lossless .ply via nerfstudio ---------------------------------------
run(f"ns-export gaussian-splat --load-config '{CONFIG_YML}' "
    f"--output-dir '{EXPORT_DIR}'",
    "Exporting Gaussian splat (.ply)", log_path="/content/ns_export.log")

plys = glob.glob(os.path.join(EXPORT_DIR, "*.ply"))
if not plys:
    raise RuntimeError("Export produced no .ply, see /content/ns_export.log")
PLY_PATH = plys[0]
print("  .ply:", PLY_PATH, f"({os.path.getsize(PLY_PATH)/1e6:.1f} MB)")

# --- 2. Compact .splat for web viewers -------------------------------------
# Standard antimatter15 .splat: per-gaussian 32 bytes:
#   position (3 f32), scale (3 f32), color+opacity (4 u8), rotation quat (4 u8).
def ply_to_splat(ply_path, splat_path):
    from plyfile import PlyData
    ply = PlyData.read(ply_path)
    v = ply["vertex"]
    names = v.data.dtype.names
    xyz = np.stack([v["x"], v["y"], v["z"]], axis=1).astype(np.float32)
    scales = np.stack([v["scale_0"], v["scale_1"], v["scale_2"]], axis=1).astype(np.float32)
    scales = np.exp(scales)
    rots = np.stack([v["rot_0"], v["rot_1"], v["rot_2"], v["rot_3"]], axis=1).astype(np.float32)
    rots = rots / (np.linalg.norm(rots, axis=1, keepdims=True) + 1e-9)
    SH_C0 = 0.28209479177387814
    color = 0.5 + SH_C0 * np.stack([v["f_dc_0"], v["f_dc_1"], v["f_dc_2"]], axis=1)
    color = np.clip(color, 0, 1)
    opacity = 1.0 / (1.0 + np.exp(-np.asarray(v["opacity"])))
    rgba = np.concatenate([color, opacity[:, None]], axis=1)
    rgba = np.clip(rgba * 255, 0, 255).astype(np.uint8)
    rot_u8 = np.clip((rots * 128) + 128, 0, 255).astype(np.uint8)
    # Sort by importance (opacity * volume) so viewers stream best splats first.
    imp = opacity * np.prod(scales, axis=1)
    order = np.argsort(-imp)
    with open(splat_path, "wb") as f:
        for i in order:
            f.write(xyz[i].tobytes())
            f.write(scales[i].tobytes())
            f.write(rgba[i].tobytes())
            f.write(rot_u8[i].tobytes())
    return len(order)

SPLAT_PATH = os.path.join(EXPORT_DIR, f"{scene_name}.splat")
try:
    n = ply_to_splat(PLY_PATH, SPLAT_PATH)
    print(f"  .splat: {SPLAT_PATH}  ({n:,} gaussians, "
          f"{os.path.getsize(SPLAT_PATH)/1e6:.1f} MB)")
except Exception as e:
    SPLAT_PATH = None
    print("  (.splat conversion skipped:", e, ")")

print("\nExport complete.")

## 9. Download your splat

Bundles everything into a single `.zip` and downloads it. The archive contains:

- `*.ply` , the full-quality Gaussian splat (use this for best results).
- `*.splat` , a compact version for quick web viewing.

**View it online (no install):** open <https://superspl.at/editor> or
<https://playcanvas.com/supersplat/editor> and drag in the `.ply`.


In [ ]:
#@title Zip + download { display-mode: "form" }
import os, glob, zipfile

zip_path = os.path.join(WORK, f"{scene_name}_gaussian_splat.zip")
files_to_zip = [p for p in [PLY_PATH, SPLAT_PATH] if p and os.path.isfile(p)]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in files_to_zip:
        z.write(p, arcname=os.path.basename(p))
    # Include a short readme for the recipient.
    readme = (
        f"Scene: {scene_name}\n"
        f"Pipeline: Nerfstudio + Splatfacto ({quality_preset} preset, "
        f"{train_iterations} iters, sfm={sfm_tool})\n\n"
        "Files:\n"
        "  *.ply   - full-quality 3D Gaussian Splat (recommended)\n"
        "  *.splat - compact version for web viewers\n\n"
        "View online: https://superspl.at/editor  (drag in the .ply)\n"
    )
    z.writestr("README.txt", readme)

print("Package contents:")
for p in files_to_zip:
    print(f"  {os.path.basename(p):40s} {os.path.getsize(p)/1e6:8.1f} MB")
print(f"\nZip: {zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)")

try:
    from google.colab import files
    print("\nStarting download...")
    files.download(zip_path)
except Exception as e:
    print("\nAuto-download unavailable:", e)
    print("Open the Files panel (folder icon, left) and download:", zip_path)

## Troubleshooting & upgrades

**Common issues**

| Symptom | Fix |
|---|---|
| *No GPU* in cell 1 | Runtime -> Change runtime type -> GPU, then re-run. |
| gsplat compile error (cell 4) | Re-run cell 4 once (kernels cache), or restart runtime and run 1 to 4 again. |
| SfM registers few/no frames (cell 6) | Set `sfm_tool='hloc'` in cell 2; recapture with slower motion and more overlap. |
| `CUDA out of memory` during training | Use `balanced` preset, lower `target_frames`, or switch to an L4/A100 runtime. |
| Splat looks noisy / floaty | Increase `train_iterations`, use `high`/`max` preset, capture more overlapping views. |
| Colab disconnects on long runs | Keep the tab active; consider Colab Pro for longer sessions and better GPUs. |

**Optional upgrade for indoor geometry: DN-Splatter**

If you care most about clean walls/floors and mesh quality on indoor scenes,
[DN-Splatter](https://github.com/maturk/dn-splatter) adds monocular depth and
normal priors on top of nerfstudio. It reuses the exact dataset produced in
cell 6 (`DATA_DIR`), so you can install it and run:

```bash
pip install git+https://github.com/maturk/dn-splatter
# generate mono depth/normal priors, then:
ns-train dn-splatter --data DATA_DIR ...   # see its README for current flags
```

It is heavier to install and more sensitive to versions, which is why the main
notebook uses Splatfacto for reliability.

---

### Credits

- **Nerfstudio** , Tancik et al., the framework and `ns-process-data` / `ns-train` / `ns-export` tooling.
- **gsplat** , Ye et al., the CUDA Gaussian rasterization backend (incl. MCMC).
- **3D Gaussian Splatting** , Kerbl et al. (SIGGRAPH 2023), the original method.
- **COLMAP** , Schönberger & Frahm, Structure-from-Motion.
- **DN-Splatter** , Turkulainen et al. (WACV 2025), optional indoor upgrade.
